# TradaBoostR2

This notebook runs TradaBoostR2 as implemented in https://adapt-python.github.io/adapt/generated/adapt.utils.make_regression_da.html. 

Make sure to install adapt package (preferrably with Python 3.9), along with Tensorflow == 2.15. 

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.model_selection import train_test_split

from adapt.instance_based import TrAdaBoostR2, TwoStageTrAdaBoostR2
from sklearn.metrics import mean_squared_error, mean_absolute_error

import itertools

from sklearn.preprocessing import StandardScaler

from adapt.instance_based import TrAdaBoostR2

from lineartree import LinearTreeRegressor
from sklearn.linear_model import LinearRegression

import warnings
warnings.filterwarnings("ignore")

In [ ]:
train_size_list = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0] #0.8or 0.93
target_columns = ['Volume', 'Dgv']
seed_list = [1,2,3,4,5]

In [3]:
predictor_columns = ['pzabovezmean', 'pzabove2', 'zq5', 'zq10',
    'zq15', 'zq20', 'zq25', 'zq30', 'zq35', 'zq40', 'zq45', 'zq50', 'zq55',
    'zq60', 'zq65', 'zq70', 'zq75', 'zq80', 'zq85', 'zq90', 'zq95',
    'zpcum1', 'zpcum2', 'zpcum3', 'zpcum4', 'zpcum5', 'zpcum6', 'zpcum7',
    'zpcum8', 'zpcum9'
    ]

In [ ]:
#ablation study for TradaBoostR2, Gaussian errors, with gaussian source domain errors
ablation_transfer_tradaboost_normal_normal = pd.DataFrame(columns = ['seed', 'target_column', 'target_instances', 'method',
                                   'n_estimators', 'lr', 'tree_size', 'val_rmse', 'val_mae', 'rmse', 'mae'])

n_estimators_list = [10,20,30,40,50]
lr_list = [0.1, 0.5, 1.0]
tree_size_list = [1,2,3,4,5]


# --- Step 2: Create full parameter grid ---
param_grid = list(itertools.product(
    n_estimators_list,
    lr_list,
    tree_size_list
))


for seed in seed_list:
    for train_size in train_size_list:
        for target_column in target_columns:

            #data from Svedala
            data_sweden = pd.read_csv(r'../datasets/rs_sweden.csv', index_col=[0])
            data_sweden = data_sweden[data_sweden['area_code'] == 4]
            print(len(data_sweden))


            #evaluate and rain on latvia instead (keep naming for simplicity)
            #data from latvia target
            data_latvia = pd.read_csv(r'../datasets/rs_lettland.csv', index_col=[0])
            data_latvia = data_latvia.rename(columns = {'H_AVERAGE': 'Hgv', 'D_AVERAGE': 'Dgv', 'VOLUME': 'Volume'})
            data_temp, data_test = train_test_split(data_latvia, test_size=0.25, random_state=seed)
            data_train, data_val = train_test_split(data_temp, test_size=0.333, random_state=seed)
            train_size_ = int(len(data_train)*train_size)
            data_train = data_train[0:train_size_]

            #"General" base dataset (to use for transfer)
            X_source_train = np.array(data_sweden[predictor_columns])
            y_source_train = np.array(data_sweden[target_column])

            #Specific train and test set
            X_target_train = np.array(data_train[predictor_columns])
            y_target_train = np.array(data_train[target_column])

            X_target_val = np.array(data_val[predictor_columns])
            y_target_val = np.array(data_val[target_column])

            X_target_test = np.array(data_test[predictor_columns])
            y_target_test = np.array(data_test[target_column])

            print(len(X_target_train), len(X_target_val), len(X_target_test))
            for config in param_grid:
                n_estimators, lr, tree_size = config


                method = f'TradaBoostR2'
                base_estimator = LinearTreeRegressor(
                                base_estimator=LinearRegression(),
                                max_depth=tree_size,
                                min_samples_leaf=4
                            )
                model = TrAdaBoostR2(base_estimator, n_estimators=n_estimators, lr=lr)
                model.fit(X_source_train, y_source_train, X_target_train, y_target_train)
                preds = model.predict(X_target_test)
                val_preds = model.predict(X_target_val)
                val_rmse = np.sqrt(mean_squared_error(val_preds, y_target_val))
                val_mae = mean_absolute_error(val_preds, y_target_val)
                rmse = np.sqrt(mean_squared_error(preds, y_target_test))
                mae = mean_absolute_error(preds, y_target_test)
                ablation_transfer_tradaboost_normal_normal.loc[len(ablation_transfer_tradaboost_normal_normal)] = [seed, target_column, train_size_, method, n_estimators,
                                                                                                                    lr, tree_size, val_rmse, val_mae, rmse, mae]
                ablation_transfer_tradaboost_normal_normal.to_csv(f'results/tradaboost_ablation_rs.csv')

570 665 665
Iteration 0 - Error: 0.1422
Iteration 1 - Error: 0.1475
Iteration 2 - Error: 0.1535
Iteration 3 - Error: 0.1602
Iteration 4 - Error: 0.1679
Iteration 5 - Error: 0.2248
Iteration 6 - Error: 0.2435
Iteration 7 - Error: 0.2604
Iteration 8 - Error: 0.2642
Iteration 9 - Error: 0.1879
Iteration 0 - Error: 0.1365
Iteration 1 - Error: 0.2052
Iteration 2 - Error: 0.1452
Iteration 3 - Error: 0.2180
Iteration 4 - Error: 0.2260
Iteration 5 - Error: 0.1587
Iteration 6 - Error: 0.1948
Iteration 7 - Error: 0.2252
Iteration 8 - Error: 0.2189
Iteration 9 - Error: 0.2415
Iteration 0 - Error: 0.1852
Iteration 1 - Error: 0.1907
Iteration 2 - Error: 0.2071
Iteration 3 - Error: 0.2053
Iteration 4 - Error: 0.2140
Iteration 5 - Error: 0.2156
Iteration 6 - Error: 0.2315
Iteration 7 - Error: 0.2178
Iteration 8 - Error: 0.1931
Iteration 9 - Error: 0.1961
Iteration 0 - Error: 0.1673
Iteration 1 - Error: 0.1729
Iteration 2 - Error: 0.1820
Iteration 3 - Error: 0.2092
Iteration 4 - Error: 0.2004
Iteratio

KeyboardInterrupt: 